# Building a Web Research Agent using the ReAct Pattern

## Helper Functions

In [ ]:
#---------- Helper functions and classes for Agent implementation ----------#
"""
This is a collection of helper functions and methods we are going to use in
the Agent implementation. You don't need to know the specific implementation
of these to follow the Agent code. But, if you are curious, feel free to check
them out.
"""

import re
import time

from colorama import Fore
from colorama import Style

from dataclasses import dataclass


def completions_create(client, messages: list, model: str) -> str:
    """
    Sends a request to the client's `completions.create` method to interact with the language model.

    Args:
        client (OpenAI): The OpenAI client object
        messages (list[dict]): A list of message objects containing chat history for the model.
        model (str): The model to use for generating tool calls and responses.

    Returns:
        str: The content of the model's response.
    """
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt (str): The actual content of the prompt.
        role (str): The role of the speaker (e.g., user, assistant).

    Returns:
        dict: A dictionary representing the structured prompt.
    """
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history: list, msg: str, role: str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history (list): The list representing the current chat history.
        msg (str): The message to append.
        role (str): The role type (e.g. 'user', 'assistant', 'system')
    """
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg: str):
        """Add a message to the queue.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


@dataclass
class TagContentResult:
    """
    A data class to represent the result of extracting tag content.

    Attributes:
        content (List[str]): A list of strings containing the content found between the specified tags.
        found (bool): A flag indicating whether any content was found for the given tag.
    """

    content: list[str]
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """
    Extracts all content enclosed by specified tags (e.g., <thought>, <response>, etc.).

    Parameters:
        text (str): The input string containing multiple potential tags.
        tag (str): The name of the tag to search for (e.g., 'thought', 'response').

    Returns:
        dict: A dictionary with the following keys:
            - 'content' (list): A list of strings containing the content found between the specified tags.
            - 'found' (bool): A flag indicating whether any content was found for the given tag.
    """
    # Build the regex pattern dynamically to find multiple occurrences of the tag
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"

    # Use findall to capture all content between the specified tag
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    # Return the dataclass instance with the result
    return TagContentResult(
        content=[content.strip() for content in matched_contents],
        found=bool(matched_contents),
    )



## Tool Class and functions

In [ ]:
#---------- Tool helper functions and classes ----------#
import json
from typing import Callable


def get_fn_signature(fn: Callable) -> dict:
    """
    Generates the signature for a given function.

    Args:
        fn (Callable): The function whose signature needs to be extracted.

    Returns:
        dict: A dictionary containing the function's name, description,
              and parameter types.
    """
    fn_signature: dict = {
        "name": fn.__name__,
        "description": fn.__doc__,
        "parameters": {"properties": {}},
    }
    schema = {
        k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"
    }
    fn_signature["parameters"]["properties"] = schema
    return fn_signature


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    """
    Validates and converts arguments in the input dictionary to match the expected types.

    Args:
        tool_call (dict): A dictionary containing the arguments passed to the tool.
        tool_signature (dict): The expected function signature and parameter types.

    Returns:
        dict: The tool call dictionary with the arguments converted to the correct types if necessary.
    """
    properties = tool_signature["parameters"]["properties"]

    # TODO: This is overly simplified but enough for simple Tools.
    type_mapping = {
        "int": int,
        "str": str,
        "bool": bool,
        "float": float,
        "list": list,
        "dict": dict,
        "tuple": tuple
    }

    for arg_name, arg_value in tool_call["arguments"].items():
        expected_type = properties[arg_name].get("type")

        if not isinstance(arg_value, type_mapping[expected_type]):
            tool_call["arguments"][arg_name] = type_mapping[expected_type](arg_value)

    return tool_call


class Tool:
    """
    A class representing a tool that wraps a callable and its signature.

    Attributes:
        name (str): The name of the tool (function).
        fn (Callable): The function that the tool represents.
        fn_signature (str): JSON string representation of the function's signature.
    """

    def __init__(self, name: str, fn: Callable, fn_signature: str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        """
        Executes the tool (function) with provided arguments.

        Args:
            **kwargs: Keyword arguments passed to the function.

        Returns:
            The result of the function call.
        """
        return self.fn(**kwargs)


def tool(fn: Callable):
    """
    A decorator that wraps a function into a Tool object.

    Args:
        fn (Callable): The function to be wrapped.

    Returns:
        Tool: A Tool object containing the function, its name, and its signature.
    """

    def wrapper():
        fn_signature = get_fn_signature(fn)
        return Tool(
            name=fn_signature.get("name"), fn=fn, fn_signature=json.dumps(fn_signature)
        )

    return wrapper()


## How the LLM is used (planning / reasoning step)

- The LLM is called only in the planning step via `generate_research_questions()`.
- The prompt asks the model to act as an expert researcher and produce a small
  list of focused research questions for the provided topic.
- This output is the LLM's "reasoning" result: instead of directly returning final
  answers, the LLM structures the problem into sub-questions that guide the agent's
  subsequent web-based information gathering (the Acting phase).

## How the reasoning step works

1. Prompting: we provide an explicit instruction to the model asking for N concise,
   well-structured questions about the topic.
2. Model output: the LLM returns text (typically numbered or bulleted). This text
   represents the LLM's decomposition of the research task — i.e., its internal
   reasoning about what aspects are important to investigate.
3. Parsing: the program splits the model output into separate lines, strips
   numbering/bullets, and stores the resulting questions for the Acting step.

This pattern follows the Planning → Acting pattern: reasoning (LLM) produces a
plan (questions) and the agent executes the plan (web searches) to gather
information.

## Code and flow explanation

- `generate_research_questions(topic, client, model, n_questions=5)`
  - Builds a concise prompt asking the LLM to produce `n_questions` research
    questions for `topic`.
  - Calls `completions_create()` (the helper) to interact with the LLM client.
  - Parses the response into a list of question strings.

- `web_search(query, max_results=3)`
  - Attempts to use `tavily` (if installed) to run a structured search and
    normalize the results.
  - If `tavily` is not available, falls back to a DuckDuckGo HTML scrape using
    `requests` + `BeautifulSoup` and returns a list of `{title, snippet}` dicts.

- `compile_report(topic, questions, search_results)`
  - Formats the gathered information into a Markdown report with sections for
    each research question and its collected snippets.

- `main()`
  - (Optional) Initializes or expects an LLM client to be wired into the
    `completions_create` helper.
  - Prompts the user for a topic and runs the pipeline:
    1. Generate questions (LLM reasoning / planning).
    2. For each question, run `web_search` (Acting) to collect web snippets.
    3. Compile the final Markdown report and save it to `output_report.md`.

In [1]:
import re
import json
import os
from openai import OpenAI

from colorama import Fore

from helper import build_prompt_structure, completions_create, ChatHistory, update_chat_history, Tool, extract_tag_content, validate_arguments

BASE_SYSTEM_PROMPT = ""

REACT_SYSTEM_PROMPT = """
You are a Web Research Agent that follows the ReAct (Reasoning + Acting) pattern.

You operate in an iterative loop with the following phases:
1. Planning – Generate research questions.
2. Acting – Use web search tools to gather information.
3. Reporting – Compile a structured research report.

If action plan involves: Generate five questions.
Enclose the questions in <question></question> tags. And then use web search tool to find answers for each question.

For every step, you MUST follow this structure:

<reasoning_summary>
Briefly explain your reasoning for the current step (1–2 sentences).
</reasoning_summary>

<action_plan>
Clearly state what you plan to do next (1–2 sentences).
</action_plan>

OPTIONAL: Tool usage (only when external information is required)

<tool_call>
{"name": "<tool_name>", "arguments": <args_dict>, "id": <monotonically-increasing-id>}
</tool_call>

You will receive tool outputs in the following format:

<observation>
{<id>: <tool_result>}
</observation>

You should use observations to refine your reasoning and decide the next action.

Here are the available tools:

<tools>
%s
</tools>

--------------------------------------------------
AGENT BEHAVIOR RULES
--------------------------------------------------

1. Planning Phase:
- Given a user-defined topic, generate 5 clear, well-structured research questions.
- Questions should cover different dimensions of the topic (background, causes, impact, trends, solutions, challenges).
- Output the questions inside <response> tags if no tool call is needed.

2. Acting Phase:
- For each research question, search the web using the available search tool.
- Extract concise, relevant, and recent key points from search results.
- Store findings mentally and move to the next question.
- Use one tool call at a time unless explicitly required.

3. Reporting Phase:
- Compile a structured research report with:
  - Title
  - Introduction
  - Separate sections for each research question
  - Conclusion
- The report should be clear, factual, and well-organized.
- Present the final report enclosed within <response> tags.

--------------------------------------------------
OUTPUT RULES
--------------------------------------------------

- Always include <reasoning_summary> and <action_plan> before any tool call.
- Do NOT fabricate information—use web search results when factual accuracy is required.
- If the user request does not require web research or tools, respond directly using <response> tags.
- The final answer MUST be a complete research report.

--------------------------------------------------
INVALID OUTPUT EXAMPLES
--------------------------------------------------

<!-- Missing reasoning_summary -->
<tool_call>...</tool_call>

<!-- Missing action_plan -->
<reasoning_summary>...</reasoning_summary>
<tool_call>...</tool_call>

--------------------------------------------------
VALID OUTPUT FORMAT
--------------------------------------------------

<reasoning_summary>...</reasoning_summary>
<action_plan>...</action_plan>
<question>...</question>
<question>...</question>
<question>...</question>
<question>...</question>
<question>...</question>

OR

<reasoning_summary>...</reasoning_summary>
<action_plan>...</action_plan>
<tool_call>...</tool_call>

OR

<response>Final structured research report</response>

Example:
topic: Artificial Intelligence
Output: <reasoning_summary>"Artificial Intelligence" is the topic, so we will generate 5 questions.</reasoning_summary>
        <action_plan>Generate five questions about Artificial Intelligence.</action_plan>
        <question>How will artificial intelligence impact the job market in the next decade?</question>
        <question>What ethical considerations should be taken into account when developing AI technologies?</question>
        <question>How can AI be leveraged to solve global challenges such as climate change?</question>
        <question>In what ways can AI enhance human creativity and innovation?</question>
        <question>What are the potential risks of relying too heavily on AI in decision-making processes?</question>

"""

class ReactAgent:
    """
    A class that represents an agent using the ReAct logic that interacts with tools to process
    user inputs, make decisions, and execute tool calls. The agent can run interactive sessions,
    collect tool signatures, and process multiple tool calls in a given round of interaction.

    Attributes:
        client (OpenAI): The OpenAI client used to handle model-based completions.
        model (str): The name of the model used for generating responses. Default is "gpt-4o".
        tools (list[Tool]): A list of Tool instances available for execution.
        tools_dict (dict): A dictionary mapping tool names to their corresponding Tool instances.
    """

    def __init__(
        self,
        tools: Tool | list[Tool],
        model: str = "gemini-2.5-flash", # gemini-2.0-flash, gemini-2.5-flash, gemini-3-pro-preview
        system_prompt: str = BASE_SYSTEM_PROMPT,
    ) -> None:
        self.client = OpenAI(
            api_key=os.getenv("GEMINI_API_KEY"),  # Google Gemini API key
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini base URL
        )
        self.model = model
        self.system_prompt = system_prompt
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {tool.name: tool for tool in self.tools}

    def add_tool_signatures(self) -> str:
        """
        Collects the function signatures of all available tools.

        Returns:
            str: A concatenated string of all tool function signatures in JSON format.
        """
        print("Adding tool signatures...")
        return "".join([tool.fn_signature for tool in self.tools])

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        """
        Processes each tool call, validates arguments, executes the tools, and collects results.

        Args:
            tool_calls_content (list): List of strings, each representing a tool call in JSON format.

        Returns:
            dict: A dictionary where the keys are tool call IDs and values are the results from the tools.
        """
        observations = {}
        for tool_call_str in tool_calls_content:
            tool_call = json.loads(tool_call_str)
            tool_name = tool_call["name"]
            tool = self.tools_dict[tool_name]

            print(Fore.GREEN + f"\nUsing Tool: {tool_name}")

            # Validate and execute the tool call
            validated_tool_call = validate_arguments(
                tool_call, json.loads(tool.fn_signature)
            )
            print(Fore.GREEN + f"\nTool call dict: \n{validated_tool_call}")

            result = tool.run(**validated_tool_call["arguments"])
            print(Fore.GREEN + f"\nTool result: \n{result}")

            # Store the result using the tool call ID
            observations[validated_tool_call["id"]] = result

        return observations

    def run(
        self,
        user_msg: str,
        max_rounds: int = 10,
    ) -> str:
        """
        Executes a user interaction session, where the agent processes user input, generates responses,
        handles tool calls, and updates chat history until a final response is ready or the maximum
        number of rounds is reached.

        Args:
            user_msg (str): The user's input message to start the interaction.
            max_rounds (int, optional): Maximum number of interaction rounds the agent should perform. Default is 10.

        Returns:
            str: The final response generated by the agent after processing user input and any tool calls.
        """
        user_prompt = build_prompt_structure(
            prompt=user_msg, role="user", tag="topic"
        )
        if self.tools:
            self.system_prompt += (
                "\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures()
            )

        chat_history = ChatHistory(
            [
                build_prompt_structure(
                    prompt=self.system_prompt,
                    role="system",
                ),
                user_prompt,
            ]
        )
        if self.tools:
            # Run the ReAct loop for max_rounds
            for _ in range(max_rounds):

                completion = completions_create(self.client, chat_history, self.model)

                response = extract_tag_content(str(completion), "response")
                if response.found:
                    return response.content[0]

                reasoning_summary = extract_tag_content(str(completion), "reasoning_summary")
                action_plan = extract_tag_content(str(completion), "action_plan")
                tool_calls = extract_tag_content(str(completion), "tool_call")
                questions = extract_tag_content(str(completion), "question")
                print(type(questions.content))
                print(questions)
                print(tool_calls)

                update_chat_history(chat_history, completion, "assistant")
                print(chat_history)

                if reasoning_summary.found:
                    print(Fore.MAGENTA + f"\nReasoning Summary: {reasoning_summary.content[0]}")

                if action_plan.found:
                    print(Fore.WHITE + f"\nAction Plan: {action_plan.content[0]}")
                
                if questions.found:
                    for q in questions.content:
                        print(Fore.CYAN + f"\nGenerated Question: {q}")
                    questions_list = [q for q in questions.content]
                    update_chat_history(chat_history, f"List of questions are: {questions_list}", "user")

                if tool_calls.found:
                    observations = self.process_tool_calls(tool_calls.content)
                    print(Fore.BLUE + f"\nObservations: {observations}")
                    update_chat_history(chat_history, f"{observations}", "user")

        return completions_create(self.client, chat_history, self.model)


In [2]:
import os
import json
import requests
from helper import tool

@tool
def search_web(questions:list[str]) -> dict:
    """
    Searches the web for information related to the given question using the Tavily API.
    Args:
        questions (list[str]): A list of questions to search for.
    Returns:
        dict: A dictionary containing the search results for each question. The dictionary has the following structure:
            {
                "success": bool,
                "responses": [
                    {
                        "question": str,
                        "answer": str
                    },
                    ...
                ],
                "message": str
            }
    """
    try:
        api_key = os.getenv("TAVILY_API_KEY")  # Replace with your actual Tavily API key
        url = "https://api.tavily.com/search"
        headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
        }
        responses = []
        for question in questions:
            payload = {
            "query": question,
            "search_depth": "basic",  # Use "advanced" for more comprehensive results
            "max_results": 5  # Adjust the number of results as needed
            }
            response = requests.post(url, headers=headers, json=payload)
            response.raise_for_status()  # Raise an exception for HTTP errors
            results = response.json()
            answer = "".join([item['content'] + "\n" if item["score"] >= 0.90 else "" for item in results.get('results', [])]).strip()
            responses.append({
                "question": question,
                "answer": answer
            })
        return {
            "success": True,
            "responses": responses,
            "message": "Web search completed successfully."
        }
    except Exception as e:
        return {
            "success": False,
            "responses": [],
            "message": f"Error searching web: {str(e)}"
        }

In [4]:
agent = ReactAgent(tools=[search_web])

In [ ]:
output = agent.run(
    user_msg="Mango"
)
output

Adding tool signatures...
<class 'list'>
TagContentResult(content=['What is the origin and historical significance of mangoes, and how have they spread globally?', 'What are the key nutritional benefits and health properties of mangoes?', 'What are the primary challenges and best practices in mango cultivation and harvesting?', 'What is the global economic impact of the mango industry, including major producing and consuming regions?', 'What cultural significance do mangoes hold in different societies, and what are some popular culinary uses?'], found=True)
TagContentResult(content=[], found=False)
[{'role': 'system', 'content': '\n\nYou are a Web Research Agent that follows the ReAct (Reasoning + Acting) pattern.\n\nYou operate in an iterative loop with the following phases:\n1. Planning – Generate research questions.\n2. Acting – Use web search tools to gather information.\n3. Reporting – Compile a structured research report.\n\nIf action plan involves: Generate five questions.\nEncl

'# The Magnificent Mango: A Global Fruit\n\n## Introduction\nThe mango, often hailed as the "King of Fruits," is a stone fruit native to South Asia, celebrated for its sweet, juicy flesh and vibrant flavor. Beyond its delicious taste, mangoes hold significant cultural, nutritional, and economic importance across the globe. This report explores the various facets of this beloved fruit, from its ancient origins and health benefits to its cultivation challenges and global market impact.\n\n## 1. Origin and Historical Significance of Mangoes, and Their Global Spread\nMangoes originated in South Asia, particularly in the region between northwestern Myanmar, Bangladesh, and northeastern India, approximately 4,000 to 5,000 years ago. Their cultivation dates back to ancient times, with archaeological evidence suggesting their presence in India as early as 2000 BCE. Historically, mangoes held significant spiritual and cultural value in India, often associated with prosperity and love in Hindu t

In [ ]:

from IPython.display import display_markdown
display_markdown(output, raw=True)

# The Magnificent Mango: A Global Fruit

## Introduction
The mango, often hailed as the "King of Fruits," is a stone fruit native to South Asia, celebrated for its sweet, juicy flesh and vibrant flavor. Beyond its delicious taste, mangoes hold significant cultural, nutritional, and economic importance across the globe. This report explores the various facets of this beloved fruit, from its ancient origins and health benefits to its cultivation challenges and global market impact.

## 1. Origin and Historical Significance of Mangoes, and Their Global Spread
Mangoes originated in South Asia, particularly in the region between northwestern Myanmar, Bangladesh, and northeastern India, approximately 4,000 to 5,000 years ago. Their cultivation dates back to ancient times, with archaeological evidence suggesting their presence in India as early as 2000 BCE. Historically, mangoes held significant spiritual and cultural value in India, often associated with prosperity and love in Hindu traditions.

The spread of mangoes from their native land was primarily facilitated by human migration and trade. Buddhist monks are credited with carrying mangoes to Southeast Asia in the 4th and 5th centuries BCE. Later, Persian traders introduced them to the Middle East and East Africa. Portuguese explorers were instrumental in bringing mangoes to Brazil in the 16th century and subsequently to other parts of the Americas. Today, mangoes are cultivated in tropical and subtropical regions worldwide, demonstrating their successful global propagation.

## 2. Key Nutritional Benefits and Health Properties of Mangoes
Mangoes are a powerhouse of nutrition, offering a wide array of vitamins, minerals, and fiber with a relatively low-calorie count. A single cup (165 grams) of fresh mango provides nearly 67% of the Daily Value (DV) for Vitamin C, a powerful antioxidant crucial for immune health, skin vitality, and iron absorption.

Key nutritional benefits and health properties include:
*   **Rich in Antioxidants:** Mangoes contain protective antioxidants that combat free radicals, potentially reducing the risk of chronic diseases.
*   **Boosts Immunity:** High Vitamin C content strengthens the immune system.
*   **Aids Digestion:** Their fiber and water content help prevent constipation and promote a healthy digestive tract.
*   **Supports Eye Health:** Mangoes are a good source of Vitamin A and other nutrients vital for maintaining healthy vision.
*   **Promotes Skin and Hair Health:** Vitamins A and C contribute to healthy skin and hair by aiding collagen production and protecting cells from damage.
*   **Potential Anticancer Effects:** Some studies link mangoes and their nutrients to potential anticancer effects due to their antioxidant compounds.

While fresh mangoes are highly nutritious, dried mangoes, though still beneficial, should be consumed in moderation due to their higher calorie density and sugar content.

## 3. Primary Challenges and Best Practices in Mango Cultivation and Harvesting
Mango cultivation, while generally robust, faces several challenges:
*   **Pest and Disease Management:** Mango trees are susceptible to various pests (e.g., fruit flies, mealybugs) and diseases (e.g., anthracnose, powdery mildew) that can significantly reduce yield and quality.
*   **Water Management:** Although somewhat drought-tolerant, consistent water supply is crucial for optimal growth, flowering, fruit set, and quality, especially in dry regions.
*   **Nutrient Management:** Proper fertilization and nutrient balance are essential for healthy growth, fruit yield, and quality, requiring careful monitoring.
*   **Climate Sensitivity:** Mango trees need full sunlight and specific temperature ranges for optimal flowering and fruiting. Extreme weather events like unseasonal rains or heatwaves can disrupt cycles.

Best practices for successful mango cultivation and harvesting include:
*   **Integrated Pest Management (IPM):** Regular monitoring and implementing IPM strategies minimize pest and disease damage.
*   **Efficient Water Scheduling:** Providing consistent water, particularly during critical growth stages, is vital.
*   **Optimized Nutrient Management:** Tailoring fertilization programs based on soil analysis and tree needs ensures healthy development.
*   **Timely Harvesting:** Knowing the right maturity signs (color, firmness, aroma) and using appropriate tools for harvesting minimizes post-harvest losses and ensures quality.
*   **Good Agricultural Practices:** Following standardized practices improves overall yield, fruit quality, and marketability.

## 4. Global Economic Impact of the Mango Industry, Including Major Producing and Consuming Regions
The global mango industry is a significant economic sector, generating substantial income for farmers and national economies. The global market size was estimated at USD 63.65 billion in 2023 and is projected to grow to USD 89.34 billion by 2028, with a CAGR of 7.1%. This growth is driven by increasing consumer demand, health consciousness, and the expansion of processed mango products (purees, juices, dried mangoes).

**Major Producing Countries (2022 statistics):**
1.  **India:** The undisputed leader, producing an estimated 26.3 million tons annually. India is also the largest consumer of its own mangoes.
2.  **Indonesia:** Second-leading producer with about 4.1 million tons annually.
3.  **China:** Produces around 3.8 million tons.
4.  **Pakistan:** Produces nearly 2.8 million tons.
5.  **Mexico:** Produces 2.5 million tons.
6.  **Brazil:** Produces 2.1 million tons.

**Major Exporting Countries:**
While India is the largest producer, **Mexico** is currently the largest mango exporter globally, with exports valued at $605 million in 2024. Indonesia also exports a significant amount. Other key exporters include Peru, Brazil, and Ecuador.

**Major Consuming Regions:**
Asia remains the largest consuming region, particularly India, which consumes most of its vast production. Growing demand is also observed in North America, Europe, and other regions as global trade and awareness increase.

## 5. Cultural Significance and Popular Culinary Uses of Mangoes
Mangoes hold profound cultural significance in many societies, particularly in South Asia. In India, they are often called the "King of Fruits" and symbolize love, prosperity, and fertility. The mango tree is sacred in Hinduism, and its leaves and fruit are used in religious ceremonies and festivals. In countries like Pakistan, mangoes are an integral part of traditions, festivities, and culinary delights. The elephant deity Ganesha is frequently depicted holding a ripe mango, symbolizing prosperity. Historically, mangoes were exchanged as gifts between rulers, symbolizing friendship and diplomacy.

Culinary uses of mangoes are incredibly diverse and widespread:
*   **Fresh Consumption:** The most popular way to enjoy mangoes is fresh, either eaten on its own or added to fruit salads.
*   **Desserts:** Mangoes are a staple in desserts, including mango sticky rice (Thailand), mango lassi (India), mango sorbet, ice cream, cakes, and puddings.
*   **Beverages:** Mango juice, smoothies, and milkshakes are popular refreshing drinks.
*   **Savory Dishes:** Unripe or green mangoes are used in savory applications, such as chutneys, pickles, curries, and salads (e.g., Thai green mango salad), providing a tart, tangy flavor.
*   **Processed Products:** Mango pulp, puree, dried mango slices, and preserves are widely used ingredients in food manufacturing globally.

## Conclusion
The mango is more than just a fruit; it is a cultural icon, a nutritional powerhouse, and a significant contributor to global economies. From its ancient origins in South Asia to its widespread cultivation and consumption across continents, the mango's journey reflects centuries of human interaction, trade, and appreciation for its unique qualities. Despite cultivation challenges, ongoing efforts in sustainable practices and pest management ensure the continued availability and quality of this beloved "King of Fruits," promising a bright future for its global impact and appeal.